# Turkish ABSA: public-data Colab experiment

This notebook trains a two-stage Turkish aspect-based sentiment analysis system: **aspect term extraction (ATE)** and **aspect polarity classification (APC)**. It uses public ABSA datasets, a conservative quality gate, a Turkish ModernBERT encoder, held-out evaluation, and error analysis. All executable project source is visible below; no repository upload is required.

**Status:** No new score is claimed until this notebook has run successfully. Use a GPU runtime, run cells in order, and keep the generated report bundle. `QUICK_MODE=True` checks the pipeline only and must not be reported as a benchmark.

**Data:** [ABSA-TR](https://huggingface.co/datasets/ytu-ce-cosmos/absa-tr) supplies Turkish span and polarity labels; [M-ABSA](https://huggingface.co/datasets/Multilingual-NLP/M-ABSA) supplies Turkish data across seven domains; [Clothing Products ABSA](https://huggingface.co/datasets/ebrukilic/clothing_products_ABSA) supplies category-level polarity without spans. The model starts from [ModernBERT-TR](https://huggingface.co/ytu-ce-cosmos/modernbert-tr-base). Published [ATE](https://huggingface.co/ytu-ce-cosmos/modernbert-tr-absa-extraction) and [APC](https://huggingface.co/ytu-ce-cosmos/modernbert-tr-absa-polarity) checkpoints are optional comparison baselines.

ABSA-TR annotations include model-generated labels; M-ABSA is translated and reviewed; clothing labels have a different supervision level. Scores are therefore reported **per source**. The training split is cleaned, while validation and test annotations stay untouched. No generated reviews are used in this notebook.

The public YTU ATE/APC checkpoints are relevant recent baselines, but their training mixtures and preprocessing differ from this experiment. Their published scores are not directly comparable with a new run here. The notebook validates checkpoint label maps and reports only same-code, same-split comparisons.

Dataset selection was checked against newer resources: [DimABSA 2026](https://aclanthology.org/2026.acl-long.1881.pdf) does not include Turkish and uses dimensional sentiment labels. The [201k-row Turkish ABSA upload](https://huggingface.co/datasets/STNM-NLPhoenix/turkish-absa) exposes a single training split and entity-level labels without character offsets in its public viewer. It is left out of this fixed, span-evaluated benchmark pending a separate annotation and split audit.


## 1. Runtime and experiment configuration

Select **Runtime → Change runtime type → GPU**. The notebook stores files under `/content/absa_work`. Set `QUICK_MODE=True` only to check execution. For a portfolio result, run the full configuration and report hardware, dataset counts, seed, package versions, and all held-out metrics.


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json, os, platform, subprocess, sys

PROJECT = Path('/content/turkish_absa_colab')
WORK = Path('/content/absa_work')
for folder in ('absa', 'scripts'):
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
(PROJECT / 'scripts' / '__init__.py').touch()
sys.path.insert(0, str(PROJECT))
os.environ['ABSA_WORK'] = str(WORK)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

QUICK_MODE = False
SEED = 42
EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACCUM = 4
MAX_LEN = 256
RUN_BASELINE = True
EVAL_IMPLICIT_LIMIT = 120
print('Python:', sys.version.split()[0], '| platform:', platform.platform())


## 2. Project source

These cells write the exact source used by the experiment. The Colab runner is derived from the repository runner and contains only public-data stages. Source files remain editable inside the runtime.


In [ ]:
%%writefile /content/turkish_absa_colab/requirements.txt
transformers>=4.48.0,<5.0.0
tokenizers>=0.21
huggingface_hub>=0.25.0,<1.0.0
accelerate>=0.34
scikit-learn>=1.4.0
pandas
matplotlib>=3.8
pyarrow
numpy
requests
fastapi
uvicorn
pydantic>=2


In [ ]:
%%writefile /content/turkish_absa_colab/absa/__init__.py



In [ ]:
%%writefile /content/turkish_absa_colab/absa/config.py
"""Configuration for Turkish Aspect-Based Sentiment Analysis."""
import json
import os
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
WORK = Path(os.environ.get("ABSA_WORK", ROOT / "work"))
DATA = WORK / "data"
SPLITS = DATA / "splits"
CACHE = WORK / "hf_cache"
MODELS = WORK / "models"
REPORTS = WORK / "reports"
LOGS = WORK / "logs"
EXPORT = WORK / "export"
for _p in (SPLITS, CACHE, MODELS, REPORTS, LOGS, EXPORT):
    _p.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "ytu-ce-cosmos/modernbert-tr-base"
BASELINE_ATE = "ytu-ce-cosmos/modernbert-tr-absa-extraction"
BASELINE_APC = "ytu-ce-cosmos/modernbert-tr-absa-polarity"


def model_revision(path):
    """Return a run-pinned Hugging Face SHA for remote model repositories."""
    return json.loads(os.environ.get("ABSA_MODEL_REVISIONS", "{}")).get(str(path))




# ---- ATE (Aspect Term Extraction) BIO labels --------------------------------
ATE_BIO = ["O", "B-ASP", "I-ASP"]
ATE_L2I = {l: i for i, l in enumerate(ATE_BIO)}
ATE_I2L = {i: l for l, i in ATE_L2I.items()}

# ---- APC (Aspect Polarity Classification) labels ----------------------------
APC_LABELS = ["negative", "neutral", "positive", "not_mentioned"]
APC_L2I = {l: i for i, l in enumerate(APC_LABELS)}
APC_I2L = {i: l for l, i in APC_L2I.items()}

POL_MAP = {
    "negative": "negative", "negatif": "negative", "neg": "negative",
    "neutral": "neutral", "nötr": "neutral", "notr": "neutral", "neu": "neutral",
    "positive": "positive", "pozitif": "positive", "pos": "positive",
}

# ---- Public HF datasets -----------------------------------------------------
ABSA_TR_REPO = "ytu-ce-cosmos/absa-tr"
ABSA_TR_FILES = {"train": "train.jsonl", "val": "dev.jsonl", "test": "test.jsonl"}

M_ABSA_REPO = "Multilingual-NLP/M-ABSA"
M_ABSA_DOMAINS = ["phone", "laptop", "restaurant", "hotel", "food", "coursera", "sight"]
# Candidate aspects used for explicit+implicit classification. A 4th APC class
# (not_mentioned) is essential: neutral sentiment does not imply absent aspect.
IMPLICIT_CANDIDATES = ["ürün", "fiyat", "teslimat", "paketleme", "kalite", "performans", "batarya", "kamera", "ekran", "ses", "iade", "müşteri hizmetleri", "beden", "kumaş"]
ASPECT_ALIASES = {
    "kargo":"teslimat", "kargosu":"teslimat", "kargo firması":"teslimat",
    "teslimatı":"teslimat", "teslimat hızı":"teslimat",
    "bataryası":"batarya", "pil ömrü":"batarya", "şarj süresi":"batarya",
    "kamera":"kamera", "kameranın":"kamera", "kamerası":"kamera",
    "ekranı":"ekran", "ekran parlaklığı":"ekran",
    "ses kalitesi":"ses", "hoparlör":"ses", "fiyatlandırma":"fiyat",
    "kumaş kalitesi":"kumaş", "bedeni":"beden", "kalıbı":"beden",
    "teslimat süresi": "teslimat", "kargo süresi": "teslimat", "şarj": "batarya",
    "pil": "batarya", "pil ömrü": "batarya", "kamerası": "kamera", "fiyatı": "fiyat",
    "ürün kalitesi": "kalite", "kumaşı": "kumaş", "müşteri temsilcisi": "müşteri hizmetleri",
    "geri ödeme": "iade", "iade süreci": "iade", "kutusu": "paketleme",
}


In [ ]:
%%writefile /content/turkish_absa_colab/absa/text.py
"""Turkish-aware normalization shared by data preparation and inference."""
import unicodedata


def tr_lower(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("I", "ı").replace("İ", "i").lower().replace("\u0307", "")


In [ ]:
%%writefile /content/turkish_absa_colab/absa/data.py
"""Dataset download, parsing (ABSA-TR + M-ABSA Turkish), span alignment, and leakage-safe splits.

Unified document schema:
{
  "id": str,
  "text": str,
  "domain": str,
  "source": str,
  "register": str,          # "formal" | "informal" | "gen_z" | "sarcastic" | "standard"
  "flags": [str, ...],      # ["irony", "idiom", "gen_z", "contrast", "negation", "implicit", ...]
  "aspects": [
    {"span": str|None, "start": int|None, "end": int|None, "aspect": str, "polarity": "negative"|"neutral"|"positive"}
  ]
}
"""
import ast
import hashlib
import json
import os
import re
from bisect import bisect_right
from collections import Counter
from pathlib import Path

from .config import ABSA_TR_FILES, ABSA_TR_REPO, APC_L2I, ATE_L2I, CACHE, M_ABSA_DOMAINS, M_ABSA_REPO, POL_MAP
from .text import tr_lower


ALIGNMENT_STATS = Counter()


def fetch(repo, filename):
    from huggingface_hub import hf_hub_download
    revisions = json.loads(os.environ.get("ABSA_DATASET_REVISIONS", "{}"))
    return hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset",
                           revision=revisions.get(repo), cache_dir=str(CACHE))


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def write_jsonl(path, rows):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def text_hash(text):
    return hashlib.md5(re.sub(r"\s+", " ", tr_lower(text)).strip().encode()).hexdigest()


def bucket(key, salt="absa"):
    return int(hashlib.md5((salt + key).encode()).hexdigest()[:8], 16) % 1000


def locate_spans(text, aspects, diagnostics=None):
    """Resolve offsets; reject a document if any explicit annotation cannot align."""
    used = []
    out = []
    for a in aspects:
        pol = POL_MAP.get(str(a.get("polarity", "")).lower().strip())
        if not pol:
            if diagnostics is not None:
                diagnostics.append({"reason": "unsupported_polarity", "polarity": a.get("polarity")})
            return []
        surf = a.get("span")
        asp_name = (a.get("aspect") or surf or "genel").strip()
        s, e = a.get("start"), a.get("end")
        if surf is None or str(surf).strip() == "" or str(surf).lower() == "null":
            out.append({"span": None, "start": None, "end": None, "aspect": asp_name, "polarity": pol})
            continue
        surf = str(surf).strip()
        if isinstance(s, int) and isinstance(e, int) and 0 <= s < e <= len(text) and text[s:e] == surf:
            used.append((s, e))
            out.append({"span": surf, "start": s, "end": e, "aspect": asp_name, "polarity": pol})
            continue
        found = None
        for m in re.finditer(re.escape(surf), text):
            if all(m.end() <= u0 or m.start() >= u1 for u0, u1 in used):
                found = (m.start(), m.end())
                break
        if found is None:
            for m in re.finditer(re.escape(surf), text, re.I):
                if all(m.end() <= u0 or m.start() >= u1 for u0, u1 in used):
                    found = (m.start(), m.end())
                    surf = text[found[0]:found[1]]
                    break
        if found is not None:
            used.append(found)
            out.append({"span": surf, "start": found[0], "end": found[1], "aspect": asp_name, "polarity": pol})
        else:
            if diagnostics is not None:
                diagnostics.append({"span": surf, "aspect": asp_name})
            return []
    return out


def parse_absa_tr(split):
    path = fetch(ABSA_TR_REPO, ABSA_TR_FILES[split])
    docs = []
    for i, r in enumerate(read_jsonl(path)):
        text = r.get("text", "").strip()
        if not text:
            continue
        ALIGNMENT_STATS[f"absa_tr_{split}_attempted_docs"] += 1
        issues = []
        asps = locate_spans(text, r.get("aspects", []), diagnostics=issues)
        if issues:
            ALIGNMENT_STATS[f"absa_tr_{split}_unaligned_docs"] += 1
        if not asps:
            continue
        flags = list(r.get("flags") or [])
        if any(a["span"] is None for a in asps) and "implicit" not in flags:
            flags.append("implicit")
        docs.append({
            "id": f"absatr_{split}_{r.get('pool_id', i)}",
            "text": text,
            "domain": r.get("domain", "general"),
            "source": "absa_tr",
            "register": "standard",
            "flags": flags,
            "aspects": asps,
        })
    return docs


def parse_m_absa(split):
    hf_split = "dev" if split == "val" else split
    docs = []
    for dom in M_ABSA_DOMAINS:
        try:
            path = fetch(M_ABSA_REPO, f"{dom}/tr/{hf_split}.txt")
        except Exception as exc:
            raise RuntimeError(f"M-ABSA missing {dom}/tr/{hf_split}.txt; refuse silent dataset loss") from exc
        with open(path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if "####" not in line:
                    continue
                text, raw = line.strip().split("####", 1)
                text = text.lstrip("\u200e\u200f").strip()
                ALIGNMENT_STATS[f"m_absa_{dom}_{split}_attempted_docs"] += 1
                try:
                    triplets = ast.literal_eval(raw)
                except Exception:
                    ALIGNMENT_STATS[f"m_absa_{dom}_{split}_malformed_docs"] += 1
                    continue
                if not isinstance(triplets, list) or any(
                        not isinstance(item, (list, tuple)) or len(item) < 3
                        for item in triplets):
                    ALIGNMENT_STATS[f"m_absa_{dom}_{split}_malformed_docs"] += 1
                    continue
                raw_asps = []
                seen = set()
                for item in triplets:
                    sp, cat, pol = item[0], item[1], item[2]
                    key = (sp, pol)
                    if key in seen:
                        continue
                    seen.add(key)
                    raw_asps.append({"span": None if sp in ("NULL", "null", "") else sp,
                                     "aspect": str(cat).split("#")[0].lower(),
                                     "polarity": pol})
                issues = []
                asps = locate_spans(text, raw_asps, diagnostics=issues)
                if issues:
                    ALIGNMENT_STATS[f"m_absa_{dom}_{split}_unaligned_docs"] += 1
                if not asps:
                    continue
                docs.append({
                    "id": f"mabsa_{dom}_{split}_{i}",
                    "text": text,
                    "domain": dom,
                    "source": "m_absa",
                    "register": "standard",
                    "flags": ["multi_aspect"] if len(asps) > 1 else [],
                    "aspects": asps,
                })
    return docs


def parse_clothing(split):
    """Clothing ABSA: aspect CATEGORY + polarity, no span supervision.
    Reads official parquet directly via hf_hub_download + pandas to avoid
    datasets/huggingface_hub version conflicts on Kaggle.
    """
    from collections import defaultdict
    import pandas as pd
    repo = "ebrukilic/clothing_products_ABSA"
    fname = "data/test-00000-of-00001.parquet" if split == "test" else "data/train-00000-of-00001.parquet"
    path = fetch(repo, fname)
    df = pd.read_parquet(path)
    grouped = defaultdict(dict)
    for r in df.to_dict("records"):
        text = str(r.get("normalized_yorum") or "").strip()
        category = str(r.get("aspects") or "").strip().lower()
        pol = POL_MAP.get(str(r.get("polarity") or "").strip().lower())
        if len(text) < 8 or not category or not pol:
            continue
        key = text_hash(text)
        b = bucket(key, salt="clothing_2026")
        if split == "val" and b >= 120:
            continue
        if split == "train" and b < 120:
            continue
        if key not in grouped:
            grouped[key] = {
                "id": f"clothing_{split}_{key[:12]}",
                "text": text,
                "domain": "clothing",
                "source": "clothing_absa",
                "register": "standard",
                "flags": ["category_only"],
                "span_supervision": False,
                "aspects": [],
                "_pairs": {},
            }
        d = grouped[key]
        if category in d["_pairs"] and d["_pairs"][category] != pol:
            d["_pairs"][category] = "CONFLICT"
        elif category not in d["_pairs"]:
            d["_pairs"][category] = pol
    docs = []
    for d in grouped.values():
        d["aspects"] = [
            {"span": None, "start": None, "end": None, "aspect": cat, "polarity": pol}
            for cat, pol in d.pop("_pairs").items()
            if pol != "CONFLICT"
        ]
        if d["aspects"]:
            docs.append(d)
    return docs


def load_public(with_clothing=True):
    ALIGNMENT_STATS.clear()
    tr = parse_absa_tr("train") + parse_m_absa("train")
    va = parse_absa_tr("val") + parse_m_absa("val")
    te_absatr = parse_absa_tr("test")
    te_mabsa = parse_m_absa("test")
    clothes_test = []
    if with_clothing:
        # Explicit failure rather than silently claiming a third data source.
        clothes_train, clothes_val, clothes_test = (parse_clothing("train"),
                                                     parse_clothing("val"),parse_clothing("test"))
        tr += clothes_train
        va += clothes_val
    held = {text_hash(d["text"]) for d in va + te_absatr + te_mabsa + clothes_test}
    seen = set()
    clean_tr = []
    for d in tr:
        h = text_hash(d["text"])
        if h not in held and h not in seen:
            seen.add(h)
            clean_tr.append(d)
    stats = {
        "train": len(clean_tr),
        "val": len(va),
        "absa_tr_test": len(te_absatr),
        "m_absa_test": len(te_mabsa),
        "polarity_dist": dict(Counter(a["polarity"] for d in clean_tr for a in d["aspects"])),
        "implicit_count": sum(1 for d in clean_tr if d.get("span_supervision", True)
                              for a in d["aspects"] if a["span"] is None),
        "sources": dict(Counter(d["source"] for d in clean_tr)),
        "dataset_caveat": "ABSA-TR uses LLM annotations; M-ABSA Turkish is translated/human reviewed; neither is guaranteed native-ecommerce gold.",
        "alignment_counts": dict(ALIGNMENT_STATS),
    }
    stats["clothing_apc_test"] = len(clothes_test)
    stats["alignment_drop_rates"] = {
        key.removesuffix("_unaligned_docs"): count /
        max(1, ALIGNMENT_STATS[key.removesuffix("_unaligned_docs") + "_attempted_docs"])
        for key, count in ALIGNMENT_STATS.items() if key.endswith("_unaligned_docs")}
    return {"train": clean_tr, "val": va, "absa_tr_test": te_absatr,
            "m_absa_test": te_mabsa, "clothing_apc_test": clothes_test}, stats


# --------------------------------------------------------- tokenization for ATE
def _align_ate(text, spans, offsets):
    spans = sorted([s for s in spans if s[0] is not None and s[1] is not None])
    starts = [s for s, _ in spans]
    labels, prev = [], -1
    for ts, te in offsets:
        if te <= ts:
            labels.append(-100)
            continue
        while ts < te and text[ts].isspace():
            ts += 1
        if ts >= te:
            labels.append(-100)
            continue
        idx = bisect_right(starts, ts) - 1
        if idx >= 0 and not (spans[idx][0] <= ts < spans[idx][1]):
            idx = -1
        if idx < 0:
            j = bisect_right(starts, ts)
            if j < len(spans) and spans[j][0] < te and (te - spans[j][0]) * 2 >= (te - ts):
                idx = j
        if idx < 0:
            labels.append(ATE_L2I["O"])
            prev = -1
        else:
            labels.append(ATE_L2I["I-ASP" if prev == idx else "B-ASP"])
            prev = idx
    return labels


def encode_ate(tok, docs, max_len=256):
    # only docs that have at least one explicit span or are genuine review sentences
    enc = tok([d["text"] for d in docs], truncation=True, max_length=max_len,
              stride=max(1, min(64, max_len // 4)), return_overflowing_tokens=True,
              return_offsets_mapping=True)
    items = []
    owners = enc.get("overflow_to_sample_mapping", range(len(docs)))
    for owner, ids, offs in zip(owners, enc["input_ids"], enc["offset_mapping"]):
        d = docs[int(owner)]
        sp = [(a["start"], a["end"]) for a in d["aspects"] if isinstance(a.get("start"), int) and isinstance(a.get("end"), int)]
        items.append({"input_ids": ids, "labels": _align_ate(d["text"], sp, offs)})
    return items


# --------------------------------------------------------- tokenization for APC
def canonical_aspect(name):
    from .config import ASPECT_ALIASES
    k = re.sub(r"\s+", " ", tr_lower(name)).strip()
    return ASPECT_ALIASES.get(k, k)


def build_implicit_ontology(docs, min_count=2, max_per_domain=40):
    """Fit the candidate vocabulary on training annotations only, per domain."""
    counts = {}
    for d in docs:
        domain = str(d.get("domain") or "general")
        tally = counts.setdefault(domain, Counter())
        for a in d["aspects"]:
            if a.get("polarity") in APC_L2I:
                tally[canonical_aspect(a.get("aspect") or a.get("span"))] += 1
    return {domain: [name for name, count in tally.most_common(max_per_domain)
                     if name and count >= min_count]
            for domain, tally in counts.items()}


# Conservative weak-supervision guard. Exclude negatives whenever an aspect can
# plausibly be IMPLIED by the sentence, even without a literal aspect word.
NEGATIVE_CUE_STEMS = {
    "teslimat": ["sipariş", "kurye", "kapımda", "geldi", "ulaştı", "bekle", "karg", "gün içinde", "gecik"],
    "fiyat": ["para", "tl", "₺", "ucuz", "pahalı", "bedava", "kazık", "fiyat", "değer", "soygun", "kuruş"],
    "kalite": ["bozul", "kır", "dağıl", "sağlam", "dayan", "kalite", "kaliteli", "kusur", "çöp", "malzeme"],
    "performans": ["hız", "don", "kas", "yavaş", "akıcı", "performans", "çalışmıyor", "bug", "drop", "ısını"],
    "batarya": ["pil", "batarya", "şarj", "enerji", "yüzde", "günü çıkar"],
    "kamera": ["foto", "video", "çekim", "kamera", "lens", "gece mod"],
    "ekran": ["ekran", "parlak", "panel", "hz", "piksel", "dokunmat"],
    "paketleme": ["kutu", "ambalaj", "paket", "sarma", "ezik", "baloncuk"],
    "iade": ["iade", "para yattı", "geri ödeme", "değişim", "geri gönd"],
    "müşteri hizmetleri": ["destek", "temsilci", "müşteri", "satıcı", "aram", "şikayet"],
    "ses": ["ses", "bas", "hoparlör", "mikrofon", "gürültü", "fan", "teneke"],
    "beden": ["beden", "kalıp", "dar", "geniş", "bol", "küçük geldi", "büyük geldi", "xl"],
    "kumaş": ["kumaş", "doku", "pamuk", "incecik", "tüylen", "kıyafet", "dikiş"],
}


def _has_cue(text, cue):
    words = [re.escape(part) for part in cue.split()]
    pattern = r"(?<!\w)" + r"\s+".join(words) + (r"\w*(?!\w)" if len(cue) >= 4 else r"(?!\w)")
    return re.search(pattern, text, flags=re.UNICODE) is not None


def safe_negative_candidates(text, positive, candidates):
    normalized = {canonical_aspect(p) for p in positive}
    lower = tr_lower(text)
    return [c for c in candidates
            if c not in normalized and c != "ürün"
            and not _has_cue(lower, c)
            and not any(_has_cue(lower, term) for term in NEGATIVE_CUE_STEMS.get(c, []))]


def apc_views(doc, policy="category"):
    """Yield unambiguous (target, polarity, aspect) views for a text-only APC pair.

    A sentence may express opposite opinions about the same category or repeat
    the same surface with different sentiment. Without a target offset, those
    identical (text, target) inputs cannot have two labels. Keep the document
    for ATE/E2E, but exclude only its ambiguous APC views.
    """
    by_target = {}
    views = []
    for a in doc["aspects"]:
        pol = a.get("polarity")
        if pol not in APC_L2I:
            continue
        target = canonical_aspect(a.get("aspect") or a.get("span") or "ürün")
        candidates = [target]
        surf = a.get("span")
        if policy == "surface" and surf:
            candidates = [str(surf).strip()]
        elif policy == "both" and surf and tr_lower(surf).strip() != target and target not in ("", "genel"):
            candidates.append(str(surf).strip())
        for view in dict.fromkeys(candidates):
            key = tr_lower(view)
            by_target.setdefault(key, set()).add(pol)
            views.append((view, pol, a))
    seen = set()
    for view, pol, a in views:
        key = tr_lower(view)
        if len(by_target[key]) != 1 or key in seen:
            continue
        seen.add(key)
        yield view, pol, a


def encode_apc(tok, docs, max_len=256, train=False, negative_per_doc=2, seed=42,
               add_negatives=False, ontology=None, view_policy=None):
    """4-way APC with hard absent-aspect negatives (NEUTRAL != NOT_MENTIONED).
    Use normalized categories as well as explicit surface forms as positive views.
    Negative aspect candidates are sampled only from a closed list absent from
    both the annotated category set and the text surface, as a weak safeguard.
    """
    import random
    rng = random.Random(seed)
    texts, pairs, labels, meta = [], [], [], []
    for d in docs:
        present = {canonical_aspect(a.get("aspect") or a.get("span") or "ürün")
                   for a in d["aspects"] if a.get("polarity") in APC_L2I}
        for view, pol, a in apc_views(d, policy=view_policy or ("both" if train else "surface")):
            texts.append(d["text"]); pairs.append(view); labels.append(APC_L2I[pol])
            meta.append({"pair_id": len(meta), "id": d["id"], "flags": d.get("flags", []),
                         "register": d.get("register", "standard"),
                         "implicit": a.get("span") is None and d.get("span_supervision", True), "target": view})
        if (train or add_negatives) and negative_per_doc and d.get("span_supervision", True):
            # The closed ontology includes general categories. Avoid 'ürün' as an
            # absent negative when an annotated aspect belongs to that product.
            candidates = safe_negative_candidates(d["text"], present,
                                                  (ontology or {}).get(str(d.get("domain") or "general"), []))
            rng.shuffle(candidates)
            for neg in candidates[:negative_per_doc]:
                texts.append(d["text"]); pairs.append(neg)
                labels.append(APC_L2I["not_mentioned"])
                meta.append({"pair_id": len(meta), "id": d["id"], "flags": d.get("flags", []),
                             "register": d.get("register", "standard"),
                             "implicit": False, "target": neg, "negative_sample": True})
    if not texts:
        return [], []
    enc = tok(texts, pairs, truncation="only_first", max_length=max_len,
              stride=max(1, min(64, max_len // 4)), return_overflowing_tokens=True)
    owners = enc.get("overflow_to_sample_mapping", range(len(texts)))
    items = [{"input_ids": ids, "attention_mask": mask, "labels": lab}
             for ids, mask, lab in zip(enc["input_ids"], enc["attention_mask"],
                                       [labels[int(i)] for i in owners])]
    return items, [meta[int(i)] for i in owners]


In [ ]:
%%writefile /content/turkish_absa_colab/absa/quality.py
"""Conservative, offset-preserving quality checks for public ABSA training rows.

Text is never rewritten: changing review characters would invalidate gold spans.
Diagnostic flags are distinct from the small set of automatic rejection rules.
"""
import html
import re
import unicodedata
from collections import Counter, defaultdict
from .text import tr_lower


URL = re.compile(r"(?i)\b(?:https?://|www\.)\S+")
EMAIL = re.compile(r"(?i)\b[\w.+-]+@[\w.-]+\.[a-z]{2,}\b")
PHONE = re.compile(r"(?<!\d)(?:\+?90[\s.-]?)?0?5\d{2}[\s.-]?\d{3}[\s.-]?\d{2}[\s.-]?\d{2}(?!\d)")
HTML_TAG = re.compile(r"<\s*/?\s*[a-z][^>]*>", re.I)
HTML_ENTITY = re.compile(r"&(?:[a-z][a-z0-9]+|\#\d+|\#x[0-9a-f]+);", re.I)
ZERO_WIDTH = re.compile(r"[\u200b-\u200f\u2060\ufeff]")
REPEATED_PUNCT = re.compile(r"[!?.,;:…\-_=]{5,}")
REPEATED_CHAR = re.compile(r"([^\W\d_])\1{5,}", re.I | re.UNICODE)
PLACEHOLDERS = {"nan", "none", "null", "n/a", "na", "-", "...", "?"}


def canonical_text(text):
    """Normalize only for duplicate detection, never for model input or span offsets."""
    s = html.unescape(unicodedata.normalize("NFKC", str(text or "")))
    s = tr_lower(ZERO_WIDTH.sub("", s))
    return re.sub(r"\s+", " ", s).strip()


def quality_flags(text):
    text = str(text or "")
    flags = []
    for name, pattern in (("url", URL), ("email", EMAIL), ("phone_number", PHONE),
                          ("html_tag", HTML_TAG), ("html_entity", HTML_ENTITY),
                          ("zero_width", ZERO_WIDTH), ("repeated_punctuation", REPEATED_PUNCT),
                          ("elongated_word", REPEATED_CHAR)):
        if pattern.search(text):
            flags.append(name)
    if any(unicodedata.category(ch) == "Cc" and ch not in "\t\n\r" for ch in text):
        flags.append("control_character")
    if "\ufffd" in text:
        flags.append("replacement_character")
    if len(text) > 1000:
        flags.append("very_long_review")
    if text and sum(ch.isalpha() for ch in text) / len(text) < 0.25:
        flags.append("low_letter_ratio")
    if len(re.findall(r"\w+", text, re.UNICODE)) <= 2:
        flags.append("very_short_review")
    return flags


def rejection_reason(doc):
    text = doc.get("text")
    if not isinstance(text, str) or not text.strip():
        return "blank_text"
    if canonical_text(text) in PLACEHOLDERS:
        return "placeholder_text"
    if not doc.get("aspects"):
        return "no_valid_aspect"
    content = HTML_TAG.sub(" ", URL.sub(" ", EMAIL.sub(" ", text)))
    if sum(ch.isalpha() for ch in content) < 3:
        return "non_linguistic_text"
    for a in doc["aspects"]:
        span = a.get("span")
        start, end = a.get("start"), a.get("end")
        if span is None:
            if start is not None or end is not None:
                return "invalid_span"
        elif not (isinstance(start, int) and isinstance(end, int)
                  and 0 <= start < end <= len(text) and text[start:end] == span):
            return "invalid_span"
    return None


def clean_training(train, heldout):
    """Filter clear corruption and normalized overlap; leave held-out rows untouched."""
    held_keys = {canonical_text(d["text"]) for docs in heldout.values() for d in docs}
    seen = set()
    accepted = []
    dropped = Counter()
    source_before = Counter()
    source_after = Counter()
    flagged = Counter()
    example_ids = defaultdict(list)
    for d in train:
        source = d.get("source", "unknown")
        source_before[source] += 1
        for flag in quality_flags(d.get("text")):
            flagged[flag] += 1
        reason = rejection_reason(d)
        key = canonical_text(d.get("text"))
        if reason is None and key in held_keys:
            reason = "normalized_heldout_overlap"
        if reason is None and key in seen:
            reason = "normalized_train_duplicate"
        if reason:
            dropped[reason] += 1
            if len(example_ids[reason]) < 5:
                example_ids[reason].append(d.get("id"))
            continue
        seen.add(key)
        accepted.append(d)
        source_after[source] += 1
    report = {
        "policy": "Train only; no text rewriting or held-out filtering; diagnostic flags do not remove rows.",
        "counts": {"input": len(train), "accepted": len(accepted), "dropped": len(train) - len(accepted)},
        "dropped_by_reason": dict(dropped),
        "source_before": dict(source_before),
        "source_after": dict(source_after),
        "diagnostic_flags_before": dict(flagged),
        "example_ids_by_reason": dict(example_ids),
    }
    return accepted, report


def filter_near_heldout(train, heldout, threshold=0.9, min_chars=40):
    """Remove high-overlap train reviews using rare-shingle retrieval and exact Jaccard.

    This is a conservative candidate search, not a guarantee of semantic deduplication.
    Held-out documents are never changed.
    """
    def shingles(text):
        normalized = re.sub(r"[^\w\s]", "", canonical_text(text))
        return {normalized[i:i + 5] for i in range(len(normalized) - 4)}

    held_rows = [doc for split in heldout.values() for doc in split
                 if len(canonical_text(doc.get("text"))) >= min_chars]
    held_sets = [shingles(doc["text"]) for doc in held_rows]
    index = defaultdict(list)
    for i, grams in enumerate(held_sets):
        for gram in grams:
            index[gram].append(i)
    kept, matches = [], []
    for doc in train:
        normalized = canonical_text(doc.get("text"))
        if len(normalized) < min_chars:
            kept.append(doc)
            continue
        grams = shingles(normalized)
        candidates = set()
        for gram in sorted((g for g in grams if g in index), key=lambda g: len(index[g]))[:16]:
            candidates.update(index.get(gram, ()))
        best = None
        for i in candidates:
            other = held_sets[i]
            if min(len(grams), len(other)) / max(len(grams), len(other)) < threshold:
                continue
            intersection = len(grams & other)
            similarity = intersection / (len(grams) + len(other) - intersection)
            if similarity >= threshold and (best is None or similarity > best[1]):
                best = (held_rows[i].get("id"), similarity)
        if best is None:
            kept.append(doc)
        else:
            matches.append({"train_id": doc.get("id"), "heldout_id": best[0],
                            "char_5gram_jaccard": round(best[1], 4)})
    return kept, {"threshold": threshold, "min_chars": min_chars,
                  "candidate_search": "16 rarest held-out character five-grams",
                  "removed": len(matches), "examples": matches[:50]}


In [ ]:
%%writefile /content/turkish_absa_colab/absa/metrics.py
"""Metrics for Turkish ABSA:
1. ATE (Aspect Term Extraction): exact & overlap span P / R / F1.
2. APC (Aspect Polarity Classification given gold aspect): accuracy, macro-F1, per-class F1,
   and slice breakdowns (irony/sarcasm, idiom, gen_z, negation, contrast, implicit).
3. End-to-End Tuple (span, polarity) F1.
"""
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

from .config import APC_I2L, APC_LABELS, ATE_I2L


def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1


# --------------------------------------------------------- trainer metrics
def ate_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, -1)
    tp = fp = fn = 0
    for p_seq, l_seq in zip(preds, labels):
        m = l_seq != -100
        g, p = _bio_spans(l_seq[m]), _bio_spans(p_seq[m])
        tp += len(g & p)
        fp += len(p - g)
        fn += len(g - p)
    p, r, f1 = prf(tp, fp, fn)
    return {"precision": p, "recall": r, "f1": f1}


def _bio_spans(seq):
    spans, cur = set(), None
    for i, t in enumerate(seq):
        tag = ATE_I2L[int(t)]
        if tag == "B-ASP" or (tag == "I-ASP" and cur is None):
            if cur is not None:
                spans.add((cur, i))
            cur = i
        elif tag == "O":
            if cur is not None:
                spans.add((cur, i))
                cur = None
    if cur is not None:
        spans.add((cur, len(seq)))
    return spans


def apc_trainer_metrics(eval_pred, metadata=None):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    if metadata is not None:
        if len(metadata) != len(labels):
            raise ValueError("APC validation metadata does not match prediction rows")
        grouped = {}
        for row, label, info in zip(logits, labels, metadata):
            key = info["pair_id"]
            if key not in grouped:
                grouped[key] = [np.zeros_like(row, dtype=np.float64), 0, label, info]
            grouped[key][0] += row
            grouped[key][1] += 1
            if grouped[key][2] != label:
                raise ValueError("Conflicting labels across APC windows")
        logits = np.array([entry[0] / entry[1] for entry in grouped.values()])
        labels = np.array([entry[2] for entry in grouped.values()])
        metadata = [entry[3] for entry in grouped.values()]
    preds = np.argmax(logits, -1)
    acc = float(accuracy_score(labels, preds))
    mf1 = float(f1_score(labels, preds, labels=[0, 1, 2, 3], average="macro", zero_division=0))
    positive = [i for i, row in enumerate(metadata or []) if not row.get("negative_sample")]
    gold_f1 = float(f1_score(labels[positive], preds[positive], labels=[0, 1, 2],
                             average="macro", zero_division=0)) if positive else 0.0
    return {"accuracy": acc, "macro_f1": mf1, "gold_macro_f1": gold_f1,
            "gold_pairs": len(positive)}


# --------------------------------------------------------- full benchmark eval
def eval_ate(docs, pred_spans_per_doc):
    ex_tp = ex_fp = ex_fn = ov_tp = ov_fp = ov_fn = 0
    for d, preds in zip(docs, pred_spans_per_doc):
        golds = [(a["start"], a["end"]) for a in d["aspects"] if isinstance(a.get("start"), int) and isinstance(a.get("end"), int)]
        gs, ps = set(golds), set(preds)
        ex_tp += len(gs & ps)
        ex_fp += len(ps - gs)
        ex_fn += len(gs - ps)
        used = set()
        for g in golds:
            for j, p in enumerate(preds):
                if j not in used and min(g[1], p[1]) > max(g[0], p[0]):
                    used.add(j)
                    ov_tp += 1
                    break
            else:
                ov_fn += 1
        ov_fp += len(preds) - len(used)
    ep, er, ef1 = prf(ex_tp, ex_fp, ex_fn)
    op, orc, of1 = prf(ov_tp, ov_fp, ov_fn)
    return {
        "exact": {"precision": ep, "recall": er, "f1": ef1},
        "overlap": {"precision": op, "recall": orc, "f1": of1},
    }


def eval_apc(y_true, y_pred, meta=None):
    acc = float(accuracy_score(y_true, y_pred)) if len(y_true) else 0.0
    mf1 = float(f1_score(y_true, y_pred, labels=[0,1,2], average="macro", zero_division=0)) if len(y_true) else 0.0
    p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2], zero_division=0)
    per_class = {
        APC_I2L[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f1[i]), "support": int(sup[i])}
        for i in range(3)
    }
    slices = {}
    if meta:
        for tag in ("irony", "idiom", "gen_z", "negation", "contrast", "implicit", "formal"):
            idx = [i for i, m in enumerate(meta) if tag in m.get("flags", []) or m.get("register") == tag or (tag == "implicit" and m.get("implicit"))]
            if len(idx) >= 5:
                yt = [y_true[i] for i in idx]
                yp = [y_pred[i] for i in idx]
                slices[tag] = {
                    "n": len(idx),
                    "acc": float(accuracy_score(yt, yp)),
                    "macro_f1": float(f1_score(yt, yp, labels=[0,1,2], average="macro", zero_division=0)),
                }
    return {"n": len(y_true), "accuracy": acc, "macro_f1": mf1, "per_class": per_class, "slices": slices}


def eval_e2e(docs, pred_tuples_per_doc):
    """End-to-end tuple (start, end, polarity) exact & overlap F1 on explicit aspects."""
    ex_tp = ex_fp = ex_fn = ov_tp = ov_fp = ov_fn = 0
    for d, preds in zip(docs, pred_tuples_per_doc):
        golds = [(a["start"], a["end"], a["polarity"]) for a in d["aspects"] if isinstance(a.get("start"), int)]
        gs, ps = set(golds), set(preds)
        ex_tp += len(gs & ps)
        ex_fp += len(ps - gs)
        ex_fn += len(gs - ps)
        used = set()
        for g in golds:
            for j, p in enumerate(preds):
                if j not in used and g[2] == p[2] and min(g[1], p[1]) > max(g[0], p[0]):
                    used.add(j)
                    ov_tp += 1
                    break
            else:
                ov_fn += 1
        ov_fp += len(preds) - len(used)
    ep, er, ef1 = prf(ex_tp, ex_fp, ex_fn)
    op, orc, of1 = prf(ov_tp, ov_fp, ov_fn)
    return {"exact_f1": ef1, "overlap_f1": of1, "exact_p": ep, "exact_r": er, "overlap_p": op, "overlap_r": orc}


def eval_implicit(docs, predicted):
    """Micro tuple F1 over explicit-NULL gold aspects only.
    Zero true implicit labels are excluded from the reported recall denominator,
    but all predicted implicit labels count against precision.
    """
    from .data import canonical_aspect
    tp=fp=fn=0
    for d, preds in zip(docs,predicted):
        gold={(canonical_aspect(a.get("aspect","")),a["polarity"])
              for a in d["aspects"] if a.get("span") is None}
        guess={(canonical_aspect(a.get("aspect","")),a["polarity"])
               for a in preds if a.get("implicit")}
        tp+=len(gold & guess); fp+=len(guess-gold); fn+=len(gold-guess)
    p,r,f1=prf(tp,fp,fn)
    return {"n_docs":len(docs),"support":tp+fn,"precision":p,"recall":r,"f1":f1,
            "tp":tp,"fp":fp,"fn":fn}


In [ ]:
%%writefile /content/turkish_absa_colab/absa/train.py
"""Train ATE (TokenClassification) or APC (SequenceClassification) with ModernBERT-TR.

Designed to run one task per GPU on Kaggle 2xT4 (e.g. ATE on GPU 0 and APC on GPU 1 in parallel):
  python -m absa.train --task ate --train X.jsonl --val V.jsonl --out DIR
  python -m absa.train --task apc --train X.jsonl --val V.jsonl --out DIR
"""
import argparse
import inspect
import json
import math
import os
import random
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F
from transformers import (AutoConfig, AutoModelForSequenceClassification, AutoModelForTokenClassification,
                          AutoTokenizer, DataCollatorForTokenClassification, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

from .config import APC_I2L, APC_L2I, ATE_I2L, ATE_L2I, BASE_MODEL, model_revision
from .data import encode_apc, encode_ate, read_jsonl
from .metrics import apc_trainer_metrics, ate_trainer_metrics


class ListDS(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


class WTrainer(Trainer):
    cw = None
    label_smoothing = 0.0

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        lg = out.logits.float()
        w = self.cw.to(lg.device) if self.cw is not None else None
        loss = F.cross_entropy(
            lg.view(-1, lg.size(-1)),
            labels.view(-1),
            weight=w,
            ignore_index=-100,
            label_smoothing=self.label_smoothing,
        )
        return (loss, out) if return_outputs else loss


def _clean_cfg(cfg):
    if hasattr(cfg, "reference_compile"):
        cfg.reference_compile = False
    for attr in ("bos_token_id", "eos_token_id"):
        v = getattr(cfg, attr, None)
        if isinstance(v, int) and v >= getattr(cfg, "vocab_size", 1 << 30):
            setattr(cfg, attr, None)
    return cfg


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", required=True, choices=["ate", "apc"])
    ap.add_argument("--train", required=True)
    ap.add_argument("--val", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--model", default=BASE_MODEL)
    ap.add_argument("--epochs", type=float, default=5)
    ap.add_argument("--lr", type=float, default=3.5e-5)
    ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--max_len", type=int, default=256)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--patience", type=int, default=4)
    ap.add_argument("--max_train", type=int, default=0)
    ap.add_argument("--negative_per_doc", type=int, default=2)
    ap.add_argument("--ontology", default="", help="Training-only domain ontology JSON from prepare/clean")
    ap.add_argument("--grad_accum", type=int, default=4)
    a = ap.parse_args()
    set_seed(a.seed)

    revision = model_revision(a.model)
    tok = AutoTokenizer.from_pretrained(a.model, revision=revision)
    tr_docs = read_jsonl(a.train)
    va_docs = read_jsonl(a.val)

    if a.task == "ate":
        # train ATE on docs that have explicit spans (plus 15% implicit-only docs as negative examples)
        tr_exp = [d for d in tr_docs if any(isinstance(x.get("start"), int) for x in d["aspects"])]
        tr_imp = [d for d in tr_docs if d.get("span_supervision", True)
                  and not any(isinstance(x.get("start"), int) for x in d["aspects"])]
        random.Random(a.seed).shuffle(tr_imp)
        tr_use = tr_exp + tr_imp[: max(1, len(tr_exp) // 8)]
        random.Random(a.seed).shuffle(tr_use)
        tr = encode_ate(tok, tr_use, a.max_len)
        va = encode_ate(tok, [d for d in va_docs if d.get("span_supervision", True)], a.max_len)
        cfg = _clean_cfg(AutoConfig.from_pretrained(a.model, revision=revision, num_labels=len(ATE_L2I), id2label=ATE_I2L, label2id=ATE_L2I))
        try:
            model = AutoModelForTokenClassification.from_pretrained(a.model, revision=revision, config=cfg, attn_implementation="sdpa")
        except Exception:
            model = AutoModelForTokenClassification.from_pretrained(a.model, revision=revision, config=cfg)
        collator = DataCollatorForTokenClassification(tok, padding=True)
        metric_fn = ate_trainer_metrics
        best_metric = "f1"
        c = Counter(l for it in tr for l in it["labels"] if l != -100)
        o = max(c[0], 1)
        cw = torch.tensor([1.0, min(5.0, (o / max(c[1], 1)) ** 0.35), min(5.0, (o / max(c[2], 1)) ** 0.35)], dtype=torch.float32)
        smoothing = 0.0
    else:
        if not a.ontology:
            raise ValueError("APC training requires --ontology built from cleaned training data")
        with open(a.ontology, encoding="utf-8") as source:
            ontology = json.load(source)
        tr, _ = encode_apc(tok, tr_docs, a.max_len, train=True,
                           negative_per_doc=a.negative_per_doc, seed=a.seed,
                           ontology=ontology)
        va, va_meta = encode_apc(tok, va_docs, a.max_len, add_negatives=True,
                                 negative_per_doc=1, seed=a.seed+1, ontology=ontology,
                                 view_policy="surface")
        cfg = _clean_cfg(AutoConfig.from_pretrained(a.model, revision=revision, num_labels=len(APC_L2I), id2label=APC_I2L, label2id=APC_L2I))
        try:
            model = AutoModelForSequenceClassification.from_pretrained(a.model, revision=revision, config=cfg, attn_implementation="sdpa")
        except Exception:
            model = AutoModelForSequenceClassification.from_pretrained(a.model, revision=revision, config=cfg)
        collator = DataCollatorWithPadding(tok, padding=True)
        metric_fn = lambda prediction: apc_trainer_metrics(prediction, va_meta)
        best_metric = "gold_macro_f1"
        c = Counter(it["labels"] for it in tr)
        tot = max(sum(c.values()), 1)
        cw = torch.tensor([min(4.0, (tot / (len(APC_L2I) * max(c[i], 1))) ** 0.45)
                           for i in range(len(APC_L2I))], dtype=torch.float32)
        smoothing = 0.05

    if a.max_train:
        random.Random(0).shuffle(tr)
        tr = tr[:a.max_train]
    print(f"[train-{a.task}] train={len(tr)} val={len(va)}", flush=True)

    steps_per_epoch = max(1, math.ceil(len(tr) / (a.bs * a.grad_accum)))
    total_steps = max(1, int(steps_per_epoch * a.epochs))
    warmup_steps = max(1, int(total_steps * 0.06))
    ev = max(1, steps_per_epoch // 2)
    cuda = torch.cuda.is_available()
    bf16 = cuda and torch.cuda.get_device_capability()[0] >= 8

    sig = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"
    raw_kw = dict(
        output_dir=a.out,
        num_train_epochs=a.epochs,
        per_device_train_batch_size=a.bs,
        per_device_eval_batch_size=a.bs * 2,
        gradient_accumulation_steps=a.grad_accum,
        learning_rate=a.lr,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",
        fp16=cuda and not bf16,
        bf16=bf16,
        max_grad_norm=1.0,
        save_strategy="steps",
        save_steps=ev,
        eval_steps=ev,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model=best_metric,
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        seed=a.seed,
        group_by_length=True,
        dataloader_num_workers=2,
        remove_unused_columns=False,
        label_names=["labels"],
    )
    raw_kw[eval_key] = "steps"
    args = TrainingArguments(**{k: v for k, v in raw_kw.items() if k in sig})

    t_sig = inspect.signature(Trainer.__init__).parameters
    t_kw = dict(
        model=model,
        args=args,
        train_dataset=ListDS(tr),
        eval_dataset=ListDS(va),
        data_collator=collator,
        compute_metrics=metric_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=a.patience)],
    )
    if "processing_class" in t_sig:
        t_kw["processing_class"] = tok
    elif "tokenizer" in t_sig:
        t_kw["tokenizer"] = tok

    trainer = WTrainer(**t_kw)
    trainer.cw = cw
    trainer.label_smoothing = smoothing
    trainer.train()

    final = os.path.join(a.out, "final")
    trainer.save_model(final)
    tok.save_pretrained(final)
    m = trainer.evaluate()
    json.dump({"val": m, "args": vars(a), "n_train": len(tr)}, open(os.path.join(a.out, "train_summary.json"), "w"), indent=1)
    print(f"[train-{a.task}] done", m, flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/turkish_absa_colab/absa/infer.py
"""Two-stage + implicit-fallback inference for Turkish ABSA (ATE + APC)."""
import re
import time
import json
from pathlib import Path
import numpy as np

from .config import model_revision
from .data import canonical_aspect
from .text import tr_lower

_TRAIL = re.compile(r"[.,;:!?)\"']+$")


def _clean_cfg(cfg):
    if hasattr(cfg, "reference_compile"):
        cfg.reference_compile = False
    for attr in ("bos_token_id", "eos_token_id"):
        v = getattr(cfg, attr, None)
        if isinstance(v, int) and v >= getattr(cfg, "vocab_size", 1 << 30):
            setattr(cfg, attr, None)
    return cfg


class ATEPredictor:
    def __init__(self, path, device=None, fp16=True):
        import torch
        from transformers import AutoConfig, AutoModelForTokenClassification, AutoTokenizer
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        revision = model_revision(path)
        self.tok = AutoTokenizer.from_pretrained(path, revision=revision)
        cfg = _clean_cfg(AutoConfig.from_pretrained(path, revision=revision))
        self.model = AutoModelForTokenClassification.from_pretrained(path, revision=revision, config=cfg).to(self.device).eval()
        labels={i:str(cfg.id2label.get(i, cfg.id2label.get(str(i),''))).upper()
                for i in range(self.model.config.num_labels)}
        expected=('O','B-ASP','I-ASP')
        if sorted(labels.values()) != sorted(expected):
            raise ValueError(f"ATE checkpoint has unsupported label map: {labels}")
        self.label_indices = [next(i for i, v in labels.items() if v == tag) for tag in expected]
        if fp16 and self.device.startswith("cuda"):
            self.model.half()

    def extract_batch(self, texts, tau=0.45, batch_size=32, max_len=256):
        torch = self.torch
        results = []
        for b in range(0, len(texts), batch_size):
            chunk = texts[b:b + batch_size]
            enc = self.tok(chunk, truncation=True, max_length=max_len,
                           stride=max(1, min(64, max_len // 4)),
                           return_overflowing_tokens=True, padding=True,
                           return_offsets_mapping=True, return_tensors="pt")
            offs = enc.pop("offset_mapping").numpy()
            owners = enc.pop("overflow_to_sample_mapping").numpy()
            mask = enc["attention_mask"].numpy().astype(bool)
            probs = []
            for start in range(0, len(owners), batch_size):
                with torch.inference_mode():
                    logits = self.model(**{k: v[start:start+batch_size].to(self.device)
                                           for k, v in enc.items()}).logits.float()
                probs.append(torch.softmax(logits, -1).cpu().numpy()[:, :, self.label_indices])
            probs = np.concatenate(probs, axis=0)
            token_rows = [{} for _ in chunk]
            for owner, o_seq, p_seq, m_seq in zip(owners, offs, probs, mask):
                o = o_seq[m_seq]
                p = p_seq[m_seq]
                keep = o[:, 1] > o[:, 0]
                o, p = o[keep], p[keep]
                for offset, row in zip(o, p):
                    key = (int(offset[0]), int(offset[1]))
                    token_rows[int(owner)].setdefault(key, []).append(row)
            for text, rows in zip(chunk, token_rows):
                ordered = sorted(rows)
                offsets = np.array(ordered, dtype=int).reshape(-1, 2)
                probabilities = np.array([np.mean(rows[key], axis=0) for key in ordered]).reshape(-1, 3)
                results.append(self._decode_one(text, offsets, probabilities, tau))
        return results

    @staticmethod
    def _decode_one(text, offsets, probs, tau):
        spans, cur = [], None

        def close():
            if cur is not None:
                s, e = cur[0], cur[1]
                while s < e and text[s].isspace():
                    s += 1
                m = _TRAIL.search(text[s:e])
                if m:
                    e = s + m.start()
                while e > s and text[e - 1].isspace():
                    e -= 1
                if e > s:
                    spans.append({
                        "start": int(s),
                        "end": int(e),
                        "span": text[s:e],
                        "score": float(np.mean(cur[2])),
                    })

        for (ts, te), pr in zip(offsets, probs):
            asp_probability = 1.0 - float(pr[0])
            label = 1 if pr[1] >= pr[2] else 2
            if asp_probability < tau:
                close(); cur = None
            elif label == 1 or cur is None:
                close()
                cur = [int(ts), int(te), [asp_probability]]
            else:
                cur[1] = int(te)
                cur[2].append(asp_probability)
        close()
        return spans


class APCPredictor:
    def __init__(self, path, device=None, fp16=True):
        import torch
        from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        revision = model_revision(path)
        self.tok = AutoTokenizer.from_pretrained(path, revision=revision)
        cfg = _clean_cfg(AutoConfig.from_pretrained(path, revision=revision))
        self.model = AutoModelForSequenceClassification.from_pretrained(path, revision=revision, config=cfg).to(self.device).eval()
        if fp16 and self.device.startswith("cuda"):
            self.model.half()
        # Baseline checkpoint might use a different id2label order than ours.
        raw = self.model.config.id2label
        expected = {"negative", "neutral", "positive", "not_mentioned"}
        labels = {i: str(raw.get(i, raw.get(str(i), ""))).lower() for i in range(self.model.config.num_labels)}
        if set(labels.values()) not in ({"negative", "neutral", "positive"}, expected) or len(labels) != len(set(labels.values())):
            raise ValueError(f"APC checkpoint has unsupported label map: {labels}")
        self.label_names = labels
        self.absent_idx = next((i for i, val in self.label_names.items() if val == "not_mentioned"), None)

    def classify_pairs(self, texts, aspects, batch_size=64, max_len=256):
        torch = self.torch
        preds, confs, probs_all = [], [], []
        for b in range(0, len(texts), batch_size):
            t_chunk = texts[b:b + batch_size]
            a_chunk = aspects[b:b + batch_size]
            enc = self.tok(t_chunk, a_chunk, truncation="only_first", max_length=max_len,
                           stride=max(1, min(64, max_len // 4)),
                           return_overflowing_tokens=True, padding=True,
                           return_tensors="pt")
            owners = enc.pop("overflow_to_sample_mapping").numpy()
            sums = np.zeros((len(t_chunk), self.model.config.num_labels), dtype=np.float64)
            counts = np.zeros(len(t_chunk), dtype=np.int32)
            for start in range(0, len(owners), batch_size):
                with torch.inference_mode():
                    logits = self.model(**{k: v[start:start+batch_size].to(self.device)
                                           for k, v in enc.items()}).logits.float().cpu().numpy()
                for owner, row in zip(owners[start:start+batch_size], logits):
                    sums[int(owner)] += row
                    counts[int(owner)] += 1
            if np.any(counts == 0):
                raise RuntimeError("Tokenizer omitted an APC input")
            mean_logits = sums / counts[:, None]
            shifted = mean_logits - mean_logits.max(axis=-1, keepdims=True)
            exp = np.exp(shifted)
            pr = exp / exp.sum(axis=-1, keepdims=True)
            idx = pr.argmax(-1)
            for i, p_row in zip(idx, pr):
                preds.append(int(i))
                confs.append(float(p_row[i]))
                probs_all.append(p_row)
        return preds, confs, probs_all


class ABSAPipeline:
    """Two-stage ABSA with optional 4-way open-set implicit candidate probing.

    Unlike the original unconditional product fallback, infer implicit aspects
    only if a 4-class APC model explicitly distinguishes NOT_MENTIONED from
    NEUTRAL. No explicit aspect -> no forced sentiment label.
    """

    def __init__(self, ate_dir, apc_dir, device=None, tau=0.45,
                 implicit_confidence=0.78, absent_margin=0.12):
        self.ate = ATEPredictor(ate_dir, device=device)
        self.apc = APCPredictor(apc_dir, device=device)
        self.tau = tau
        self.implicit_confidence = implicit_confidence
        self.absent_margin = absent_margin
        ontology_path = Path(apc_dir) / "implicit_ontology.json"
        self.ontology = json.loads(ontology_path.read_text(encoding="utf-8")) if ontology_path.exists() else {}

    def analyze(self, text, include_implicit_fallback=True, domain=None):
        if not isinstance(text, str) or not text.strip():
            raise ValueError("text must be a non-empty string")
        t0 = time.perf_counter()
        spans = self.ate.extract_batch([text], tau=self.tau)[0]
        aspects = []
        if spans:
            targets = [s["span"] for s in spans]
            preds, confs, _ = self.apc.classify_pairs([text]*len(targets), targets)
            for s, p, conf in zip(spans, preds, confs):
                label = self.apc.label_names[p]
                if label == "not_mentioned":
                    continue
                aspects.append({
                    "span":s["span"], "start":s["start"], "end":s["end"],
                    "aspect":canonical_aspect(s["span"]), "polarity":label,
                    "confidence":round(float(conf),4), "implicit":False
                })
        # 3-way baseline APC cannot learn presence/absence, so do NOT fabricate
        # implicit-aspect labels from arbitrary sentiments.
        if include_implicit_fallback and self.apc.absent_idx is not None and domain in self.ontology:
            mentioned = {canonical_aspect(a["aspect"]) for a in aspects}
            candidates = [c for c in self.ontology[domain] if c not in mentioned]
            # A candidate named explicitly in the sentence is not implicit.
            candidates = [c for c in candidates if tr_lower(c) not in tr_lower(text)]
            pred, conf, prob = self.apc.classify_pairs([text]*len(candidates),candidates)
            choices=[]
            for cand, idx, row in zip(candidates,pred,prob):
                label=self.apc.label_names[idx]
                if label == "not_mentioned":
                    continue
                posterior=float(row[idx]); absent=float(row[self.apc.absent_idx])
                if posterior >= getattr(self,'implicit_confidence',0.78) and posterior-absent >= getattr(self,'absent_margin',0.12):
                    choices.append((cand,label,posterior))
            choices.sort(key=lambda x:x[2],reverse=True)
            # A small cap prevents numerous spurious ontology predictions.
            for cand,label,posterior in choices[:3]:
                aspects.append({"span":None,"start":None,"end":None,
                                "aspect":cand,"polarity":label,
                                "confidence":round(posterior,4),"implicit":True})
        return {"text":text,"aspects":aspects,
                "latency_ms":round((time.perf_counter()-t0)*1000,2)}


In [ ]:
%%writefile /content/turkish_absa_colab/scripts/audit_data.py
"""Offline audit of prepared public JSONL splits and label quality."""
import argparse
import json
import random
from collections import Counter, defaultdict
from pathlib import Path
import sys

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from absa.data import read_jsonl, text_hash, write_jsonl
from absa.quality import canonical_text
from absa.text import tr_lower


def collect(work):
    files={}
    for d in [work/'data'/'splits']:
        if d.exists():
            for f in sorted(d.glob('*.jsonl')):
                if f.stem in ('train_full', 'train_clean', 'val_full', 'curated_stress_test'):
                    continue
                files[f.stem]=read_jsonl(f)
    return files


def audit(files,review_n=100,seed=42):
    report={"splits":{},"duplicates":{},"normalized_duplicates":{},"invalid_spans":[],"conflicts":[],"warnings":[]}
    for name,docs in files.items():
        labels=Counter(a.get('polarity') for d in docs for a in d.get('aspects',[]))
        sources=Counter(d.get('source','unknown') for d in docs)
        flags=Counter(flag for d in docs for flag in d.get('flags',[]))
        report['splits'][name]={"rows":len(docs),"sources":dict(sources),
                                "polarities":dict(labels),"flags":dict(flags),
                                "implicit_gold":sum(1 for d in docs if d.get('span_supervision',True)
                                                    for a in d.get('aspects',[]) if a.get('span') is None),
                                "category_only":sum(1 for d in docs if not d.get('span_supervision',True))}
        for d in docs:
            seen={}
            for a in d.get('aspects',[]):
                surf=a.get('span');start=a.get('start');end=a.get('end')
                if surf is not None and not(isinstance(start,int) and isinstance(end,int)
                                            and d['text'][start:end]==surf):
                    report['invalid_spans'].append({"split":name,"id":d.get('id'),"span":surf})
                k=(tr_lower(a.get('aspect','')),surf or '<implicit>')
                if k in seen and seen[k]!=a.get('polarity'):
                    report['conflicts'].append({"split":name,"id":d.get('id'),"aspect":k})
                seen[k]=a.get('polarity')
    train_rows=files.get('train_public',[])
    train={text_hash(d['text']) for d in train_rows}
    normalized_train={canonical_text(d['text']) for d in train_rows}
    for k,v in files.items():
        if k == 'train_public': continue
        overlap=len(train & {text_hash(d['text']) for d in v})
        report['duplicates']['train_vs_'+k]=overlap
        report['normalized_duplicates']['train_vs_'+k]=len(normalized_train & {canonical_text(d['text']) for d in v})
        if overlap:report['warnings'].append(f'Potential train leakage into {k}: {overlap} exact matches')
    report['warnings'].append('Cross-split exact hashing is not semantic deduplication; evaluate semantic near-duplicates separately.')
    rand=random.Random(seed)
    samples = list(train_rows)
    rand.shuffle(samples)
    review=[{"id":d.get('id'),"text":d.get('text'),"suggested_aspects":d.get('aspects'),
             "reviewer_aspects":None,"acceptable":None,"error_type":None,
             "flags":d.get('flags',[]),"source":d.get('source')} for d in samples[:review_n]]
    return report,review


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--work',type=Path,required=True)
    p.add_argument('--review_n',type=int,default=100)
    a=p.parse_args()
    files=collect(a.work)
    report,review=audit(files,a.review_n)
    out=a.work/'reports'
    out.mkdir(parents=True,exist_ok=True)
    (out/'data_audit.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    write_jsonl(out/'review_queue.jsonl',review)
    print(json.dumps({'sources':list(files),'invalid_spans':len(report['invalid_spans']),
                      'conflicts':len(report['conflicts']), 'warnings':report['warnings']},ensure_ascii=False,indent=2))
    print('Saved',out/'data_audit.json',out/'review_queue.jsonl')


if __name__=='__main__':main()


In [ ]:
%%writefile /content/turkish_absa_colab/scripts/eda_data.py
"""Reproducible EDA for prepared Turkish ABSA splits."""
import argparse
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

from absa.data import canonical_aspect, read_jsonl
from absa.config import model_revision
from absa.quality import canonical_text, quality_flags


SPLIT_NAMES = ("train_public", "train_clean", "val", "absa_tr_test", "m_absa_test", "clothing_apc_test")


def _quantiles(values):
    if not values:
        return {"p50": 0, "p90": 0, "p95": 0, "p99": 0, "max": 0}
    s = pd.Series(values)
    return {f"p{int(q * 100)}": round(float(s.quantile(q)), 1)
            for q in (0.5, 0.9, 0.95, 0.99)} | {"max": int(s.max())}


def _token_lengths(docs, model_name, max_len):
    if not model_name:
        return {"status": "not_requested"}
    try:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(model_name, revision=model_revision(model_name), use_fast=True)
        lengths = []
        pair_lengths = []
        for i in range(0, len(docs), 256):
            batch = docs[i:i + 256]
            texts = [d["text"] for d in batch]
            targets = [str((d["aspects"][0].get("aspect") or d["aspects"][0].get("span") or ""))
                       if d.get("aspects") else "" for d in batch]
            lengths += [len(x) for x in tok(texts, truncation=False)["input_ids"]]
            pair_lengths += [len(x) for x in tok(texts, targets, truncation=False)["input_ids"]]
        return {"status": "ok", "tokenizer": model_name, "max_len": max_len,
                "ate_lengths": _quantiles(lengths), "apc_pair_lengths": _quantiles(pair_lengths),
                "ate_over_max_len": sum(n > max_len for n in lengths),
                "apc_pair_over_max_len": sum(n > max_len for n in pair_lengths),
                "n_documents": len(docs)}
    except Exception as exc:
        return {"status": "failed", "error": f"{type(exc).__name__}: {exc}"}


def run_eda(work, max_len=256, tokenizer_name=None):
    work = Path(work)
    split_dir = work / "data" / "splits"
    files = {name: read_jsonl(split_dir / f"{name}.jsonl") for name in SPLIT_NAMES
             if (split_dir / f"{name}.jsonl").exists()}
    if "train_clean" not in files or "val" not in files:
        raise FileNotFoundError("Run prepare and clean before EDA")

    summary = []
    by_source = Counter()
    by_domain = Counter()
    by_polarity = Counter()
    by_aspect = Counter()
    noise_flags = Counter()
    ambiguous = Counter()
    keys = {}
    for name, docs in files.items():
        length = [len(d["text"]) for d in docs]
        n_aspects = [len(d.get("aspects", [])) for d in docs]
        raw_keys = [canonical_text(d["text"]) for d in docs]
        keys[name] = set(raw_keys)
        summary.append({"split": name, "documents": len(docs), "aspects": sum(n_aspects),
                        "explicit_aspects": sum(a.get("span") is not None for d in docs for a in d["aspects"]),
                        "implicit_aspects": sum(a.get("span") is None and d.get("span_supervision", True)
                                                for d in docs for a in d["aspects"]),
                        "category_only_documents": sum(not d.get("span_supervision", True) for d in docs),
                        "multi_aspect_documents": sum(n > 1 for n in n_aspects),
                        "normalized_duplicate_rows": len(raw_keys) - len(keys[name]),
                        "char_p50": _quantiles(length)["p50"], "char_p95": _quantiles(length)["p95"],
                        "char_p99": _quantiles(length)["p99"],
                        "aspects_per_doc_p95": _quantiles(n_aspects)["p95"]})
        for d in docs:
            by_source[name, d.get("source", "unknown")] += 1
            by_domain[name, d.get("domain", "unknown")] += 1
            noise_flags.update((name, flag) for flag in quality_flags(d["text"]))
            targets = defaultdict(set)
            for a in d["aspects"]:
                pol = a.get("polarity", "unknown")
                by_polarity[name, pol] += 1
                aspect = canonical_aspect(a.get("aspect") or a.get("span") or "")
                by_aspect[name, aspect] += 1
                targets[aspect].add(pol)
            ambiguous[name] += sum(len(pols) > 1 for pols in targets.values())

    overlap = []
    for test_name in ("val", "absa_tr_test", "m_absa_test", "clothing_apc_test"):
        if test_name in keys:
            for train_name in ("train_public", "train_clean"):
                if train_name in keys:
                    overlap.append({"train": train_name, "heldout": test_name,
                                    "normalized_text_collisions": len(keys[train_name] & keys[test_name])})

    reports = work / "reports"
    reports.mkdir(parents=True, exist_ok=True)
    frames = {
        "summary": pd.DataFrame(summary),
        "source": pd.DataFrame([{"split": k[0], "source": k[1], "documents": v} for k, v in by_source.items()]),
        "domain": pd.DataFrame([{"split": k[0], "domain": k[1], "documents": v} for k, v in by_domain.items()]),
        "polarity": pd.DataFrame([{"split": k[0], "polarity": k[1], "aspects": v} for k, v in by_polarity.items()]),
        "aspect": pd.DataFrame([{"split": k[0], "aspect": k[1], "aspects": v} for k, v in by_aspect.items()]),
        "noise": pd.DataFrame([{"split": k[0], "flag": k[1], "documents": v} for k, v in noise_flags.items()]),
        "overlap": pd.DataFrame(overlap),
    }
    for name, frame in frames.items():
        frame.to_csv(reports / f"eda_{name}.csv", index=False)
    token = _token_lengths(files["train_clean"] + files["val"], tokenizer_name, max_len)
    report = {"files": list(files), "summary": summary,
              "ambiguous_category_polarities_by_split": dict(ambiguous),
              "normalized_cross_split_overlap": overlap,
              "token_lengths_train_clean_plus_val": token,
              "notes": ["Official validation and test texts are never cleaned or removed.",
                        "Diagnostics describe possible noise, not verified annotation errors.",
                        "Train-clean and train-public are alternative versions of the same rows; do not sum them."]}
    (reports / "eda_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({"summary": summary, "overlap": overlap, "tokens": token}, ensure_ascii=False), flush=True)
    return report


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--work", type=Path, required=True)
    parser.add_argument("--max_len", type=int, default=256)
    parser.add_argument("--tokenizer", default=None)
    args = parser.parse_args()
    run_eda(args.work, args.max_len, args.tokenizer)


In [ ]:
%%writefile /content/turkish_absa_colab/serve.py
"""FastAPI microservice for Turkish aspect extraction and polarity classification.
ABSA_ATE_DIR=work/export/ate_model ABSA_APC_DIR=work/export/apc_model uvicorn serve:app --port 8001
"""
import os
import threading
from typing import List, Optional

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator

app = FastAPI(title="Turkish ABSA API", version="1.0")
_pipe = None
_load_lock = threading.Lock()


def pipe():
    global _pipe
    if _pipe is None:
        with _load_lock:
            if _pipe is None:
                from absa.infer import ABSAPipeline
                ate_dir = os.environ.get("ABSA_ATE_DIR", "work/export/ate_model")
                apc_dir = os.environ.get("ABSA_APC_DIR", "work/export/apc_model")
                dev = os.environ.get("ABSA_DEVICE", "cpu")
                _pipe = ABSAPipeline(ate_dir, apc_dir, device=dev)
    return _pipe


class Req(BaseModel):
    text: str = Field(..., min_length=1, max_length=5000)
    include_implicit_fallback: bool = True
    domain: Optional[str] = None

    @field_validator("text")
    @classmethod
    def non_blank_text(cls, value):
        if not value.strip():
            raise ValueError("text must contain a non-whitespace character")
        return value


class AspectOut(BaseModel):
    span: Optional[str] = None
    start: Optional[int] = None
    end: Optional[int] = None
    aspect: str
    polarity: str
    confidence: float
    implicit: bool


class Resp(BaseModel):
    text: str
    aspects: List[AspectOut]
    latency_ms: float


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": _pipe is not None}


@app.post("/analyze", response_model=Resp)
def analyze(r: Req):
    model = pipe()
    try:
        return model.analyze(r.text, include_implicit_fallback=r.include_implicit_fallback,
                             domain=r.domain)
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc


In [ ]:
%%writefile /content/turkish_absa_colab/run_public.py
"""Public-data-only Colab runner, generated from run_all.py."""
import json
import os
import random
import shutil
import subprocess
import sys
import time
from collections import Counter

from absa import config as C
from absa.data import load_public, read_jsonl, write_jsonl

S = C.SPLITS


def stage_prepare(a):
    sp, stats = load_public(with_clothing=not a.no_clothing)
    written_rows = {}
    for k, v in sp.items():
        if a.quick:
            v = v[:250] if k == "train" else v[:100]
        name = "train_public" if k == "train" else k
        write_jsonl(S / f"{name}.jsonl", v)
        written_rows[name] = len(v)
    stats["written_rows"] = written_rows
    stats["quick_mode"] = bool(a.quick)
    json.dump(stats, open(S / "manifest.json", "w"), indent=2, ensure_ascii=False)
    print("[prepare]", json.dumps(stats, ensure_ascii=False), flush=True)

def stage_clean(a):
    from absa.quality import clean_training, filter_near_heldout
    from absa.data import build_implicit_ontology
    train = read_jsonl(S / "train_public.jsonl")
    held = {name: read_jsonl(S / f"{name}.jsonl") for name in
            ("val", "absa_tr_test", "m_absa_test", "clothing_apc_test")
            if (S / f"{name}.jsonl").exists()}
    cleaned, report = clean_training(train, held)
    cleaned, near_report = filter_near_heldout(cleaned, held)
    report["near_heldout"] = near_report
    report["counts"]["accepted"] = len(cleaned)
    report["counts"]["dropped"] = len(train) - len(cleaned)
    report["dropped_by_reason"]["near_heldout_overlap"] = near_report["removed"]
    report["source_after"] = dict(Counter(d.get("source", "unknown") for d in cleaned))
    write_jsonl(S / "train_clean.jsonl", cleaned)
    ontology = build_implicit_ontology(cleaned)
    (S / "implicit_ontology.json").write_text(
        json.dumps(ontology, ensure_ascii=False, indent=2), encoding="utf-8")
    (C.REPORTS / "cleaning_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[clean]", json.dumps(report["counts"], ensure_ascii=False), flush=True)

def stage_eda(a):
    from scripts.eda_data import run_eda
    run_eda(C.WORK, max_len=a.max_len, tokenizer_name=C.BASE_MODEL if a.eda_tokenizer else None)

def stage_train(a):
    import torch

    pub = read_jsonl(S / "train_clean.jsonl")
    override = getattr(a, "train_override", None)
    full_rows = read_jsonl(override) if override else pub
    random.Random(a.seed).shuffle(full_rows)
    train_path = S / "train_full.jsonl"
    write_jsonl(train_path, full_rows)

    val_rows = read_jsonl(S / "val.jsonl")
    # Validation is intentionally REAL/ORIGINAL public annotations only.
    val_path = S / "val_full.jsonl"
    write_jsonl(val_path, val_rows)

    print(f"[train] {len(full_rows)} training rows ({len(pub)} public); val={len(val_rows)}", flush=True)

    jobs = []
    variants = [("full", train_path)]
    if a.train_real_baseline:
        variants.append(("real", S / "train_clean.jsonl"))
    for variant, variant_train_path in variants:
        for task in ("ate", "apc"):
            name = f"{task}_{variant}"
            out = C.MODELS / name
            if (out / "final").exists() and not a.retrain:
                print(f"[train] {name} already trained, skipping", flush=True)
                continue
            cmd = [
                sys.executable, "-m", "absa.train",
                "--task", task,
                "--train", str(variant_train_path),
                "--val", str(val_path),
                "--out", str(out),
                "--epochs", "1" if a.quick else str(a.epochs),
                "--bs", str(a.bs),
                "--lr", str(a.lr),
                "--seed", str(a.seed),
                "--max_len", str(a.max_len),
                "--grad_accum", str(a.grad_accum),
            ]
            if task == "apc":
                cmd += ["--ontology", str(S / "implicit_ontology.json")]
            if a.quick:
                cmd += ["--max_train", "200"]
            jobs.append((name, cmd))

    ng = torch.cuda.device_count()
    print(f"[train] launching {len(jobs)} task(s) across {ng} GPU(s)", flush=True)
    i = 0
    while i < len(jobs):
        batch = jobs[i:i + max(1, ng)]
        procs = []
        for gi, (task, cmd) in enumerate(batch):
            env = {**os.environ, "TOKENIZERS_PARALLELISM": "false"}
            if ng:
                env["CUDA_VISIBLE_DEVICES"] = str(gi)
            log = open(C.LOGS / f"train_{task}.log", "w")
            procs.append((task, subprocess.Popen(cmd, cwd=str(C.ROOT), env=env, stdout=log, stderr=subprocess.STDOUT), log))
            print(f"[train] started {task.upper()} on GPU {gi if ng else 'cpu'}", flush=True)
        while any(p.poll() is None for _, p, _ in procs):
            time.sleep(45)
            for task, p, _ in procs:
                lines = open(C.LOGS / f"train_{task}.log", errors="ignore").read().replace("\r", "\n").strip().split("\n")
                tail = lines[-1] if lines else ""
                print(f"  [{task.upper()}] {tail[-150:]}", flush=True)
        for task, p, log in procs:
            log.close()
            if p.returncode != 0:
                print(open(C.LOGS / f"train_{task}.log", errors="ignore").read()[-3000:])
                raise SystemExit(f"training {task} failed")
        i += len(batch)

def _gold_apc_pairs(docs):
    from absa.data import apc_views
    texts, targets, labels, meta = [], [], [], []
    for d in docs:
        for target, pol_name, a in apc_views(d, policy="surface"):
            texts.append(d["text"])
            targets.append(target)
            labels.append(C.APC_L2I[pol_name])
            meta.append({
                "id": d["id"],
                "flags": d.get("flags", []),
                "register": d.get("register", "standard"),
                "implicit": a.get("span") is None and d.get("span_supervision", True),
            })
    return texts, targets, labels, meta

def stage_eval(a):
    from absa.infer import APCPredictor, ATEPredictor
    from absa.data import canonical_aspect
    from absa.metrics import eval_apc, eval_ate, eval_e2e

    tests = {}
    for name in ("absa_tr_test", "m_absa_test", "clothing_apc_test"):
        f = S / f"{name}.jsonl"
        if f.exists():
            tests[name] = read_jsonl(f)

    systems = {}
    if not a.skip_baseline:
        systems["ytu_baseline"] = (C.BASELINE_ATE, C.BASELINE_APC)
    if (C.MODELS / "ate_real" / "final").exists() and (C.MODELS / "apc_real" / "final").exists():
        systems["ours_real_only"] = (str(C.MODELS / "ate_real" / "final"), str(C.MODELS / "apc_real" / "final"))
    if (C.MODELS / "ate_full" / "final").exists() and (C.MODELS / "apc_full" / "final").exists():
        systems["ours_full"] = (str(C.MODELS / "ate_full" / "final"), str(C.MODELS / "apc_full" / "final"))

    report = {}
    for sys_name, (ate_path, apc_path) in systems.items():
        print(f"[eval] evaluating system: {sys_name}", flush=True)
        ate = ATEPredictor(ate_path)
        apc = APCPredictor(apc_path)
        report[sys_name] = {}
        for tname, docs in tests.items():
            # 1. ATE evaluation on explicit aspects
            span_docs = [d for d in docs if d.get("span_supervision", True)]
            raw_spans = ate.extract_batch([d["text"] for d in span_docs], tau=0.45) if span_docs else []
            ate_spans = [[(s["start"], s["end"]) for s in sp] for sp in raw_spans]
            ate_m = eval_ate(span_docs, ate_spans) if span_docs else None

            # 2. APC evaluation on gold aspects (including implicit aspects!)
            texts, targets, y_true, meta = _gold_apc_pairs(docs)
            apc_skipped = sum(1 for d in docs for asp in d["aspects"]
                              if asp.get("polarity") in C.APC_L2I) - len(y_true)
            raw_pred, _, _ = apc.classify_pairs(texts, targets)
            # Compare actual label NAMES: checkpoints may store a different
            # numeric id2label order than the locally fine-tuned classifier.
            y_pred = [C.APC_L2I.get(apc.label_names[p], -1) for p in raw_pred]
            apc_m = eval_apc(y_true, y_pred, meta=meta)

            # 3. End-to-End (extracted span + predicted polarity)
            flat_texts, flat_targets, owners = [], [], []
            for doc_idx, (d, spans) in enumerate(zip(span_docs, raw_spans)):
                for span in spans:
                    flat_texts.append(d["text"])
                    flat_targets.append(span["span"])
                    owners.append((doc_idx, span))
            flat_preds, _, _ = apc.classify_pairs(flat_texts, flat_targets)
            e2e_tuples = [[] for _ in span_docs]
            for (doc_idx, span), pred in zip(owners, flat_preds):
                label = apc.label_names[pred]
                if label != "not_mentioned":
                    e2e_tuples[doc_idx].append((span["start"], span["end"], label))
            e2e_m = eval_e2e(span_docs, e2e_tuples) if span_docs else None

            entry = {"ate": ate_m, "apc": apc_m, "apc_ambiguous_or_duplicate_skipped": apc_skipped,
                     "e2e": e2e_m}
            if a.eval_implicit_limit and apc.absent_idx is not None and span_docs:
                from absa.metrics import eval_implicit
                from absa.infer import ABSAPipeline
                # Reuse loaded predictors (do not load a third copy on GPU).
                obj = ABSAPipeline.__new__(ABSAPipeline)
                obj.ate, obj.apc, obj.tau = ate, apc, 0.45
                obj.implicit_confidence, obj.absent_margin = 0.78, 0.12
                obj.ontology = json.loads((S / "implicit_ontology.json").read_text(encoding="utf-8"))
                subset = random.Random(a.seed).sample(span_docs, min(a.eval_implicit_limit, len(span_docs)))
                pred = [obj.analyze(d["text"], domain=d.get("domain"), include_implicit_fallback=True)["aspects"]
                        for d in subset]
                entry["implicit"] = eval_implicit(subset, pred)
                entry["implicit"]["sampling"] = "uniform_document_sample"
                entry["implicit"]["positive_documents"] = sum(
                    any(asp.get("span") is None for asp in d["aspects"]) for d in subset)
                covered = sum(
                    1 for d in subset for asp in d["aspects"]
                    if asp.get("span") is None and
                    canonical_aspect(asp.get("aspect"))
                    in obj.ontology.get(str(d.get("domain") or "general"), []))
                entry["implicit"]["ontology_gold_covered"] = covered
                support = entry["implicit"]["support"]
                entry["implicit"]["ontology_gold_coverage"] = covered / support if support else None
            report[sys_name][tname] = entry
        del ate, apc
        import gc, torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    json.dump(report, open(C.REPORTS / "results.json", "w"), indent=2, ensure_ascii=False)
    md = format_markdown(report)
    (C.REPORTS / "results.md").write_text(md, encoding="utf-8")
    print(md, flush=True)

def format_markdown(report):
    L = ["# Turkish ABSA Evaluation Results (ATE + APC + Figurative/Idiom/Gen-Z Slices)\n"]
    tests = sorted({t for r in report.values() for t in r})
    L.append("## 1. Overall Benchmark Summary\n")
    for tn in tests:
        L.append(f"\n### Test Set: `{tn}`\n")
        L.append("| System | ATE Exact-F1 | ATE Overlap-F1 | APC Gold Pairs | APC Skipped | APC Accuracy | APC Macro-F1 | E2E Explicit Tuple Overlap-F1 | Implicit Category+Polarity F1 (subset) |")
        L.append("|---|---|---|---|---|---|---|---|---|")
        for sys_name, res in report.items():
            if tn not in res:
                continue
            m = res[tn]
            L.append(
                f"| **{sys_name}** | {m['ate']['exact']['f1']:.3f} | {m['ate']['overlap']['f1']:.3f} | "
                if m['ate'] else f"| **{sys_name}** | N/A | N/A | "
            )
            L[-1] += (f"{m['apc']['n']} | {m['apc_ambiguous_or_duplicate_skipped']} | "
                      f"{m['apc']['accuracy']:.3f} | **{m['apc']['macro_f1']:.3f}** | "
                      f"{m['e2e']['overlap_f1']:.3f} | " if m['e2e'] else
                      f"{m['apc']['n']} | {m['apc_ambiguous_or_duplicate_skipped']} | "
                      f"{m['apc']['accuracy']:.3f} | **{m['apc']['macro_f1']:.3f}** | N/A | ")
            implicit = m.get("implicit")
            L[-1] += f"{implicit['f1']:.3f} |" if implicit else "N/A |"
    L.append("\n## 2. Implicit probe diagnostics\n")
    L.append("| System | Test | Sampled documents | Documents with implicit gold | Gold tuples | Ontology coverage | Implicit F1 |")
    L.append("|---|---|---:|---:|---:|---:|---:|")
    for system, tests_by_name in report.items():
        for test_name, result in tests_by_name.items():
            imp = result.get("implicit")
            if imp:
                coverage = f"{imp['ontology_gold_coverage']:.3f}" if imp['ontology_gold_coverage'] is not None else "N/A"
                L.append(f"| {system} | {test_name} | {imp['n_docs']} | {imp['positive_documents']} | "
                         f"{imp['support']} | {coverage} | {imp['f1']:.3f} |")
    L.append("\n## 3. Challenging Linguistic Slices (APC Macro-F1 on Sarcasm / Idiom / Gen-Z / Negation / Implicit)\n")
    for tn in tests:
        has_slices = any(report[s].get(tn, {}).get("apc", {}).get("slices") for s in report)
        if not has_slices:
            continue
        L.append(f"\n### Slice Breakdown on `{tn}`\n")
        L.append("| System | Irony/Sarcasm | Idiom/Metaphor | Gen-Z Slang | Negation | Contrast | Implicit Aspect |")
        L.append("|---|---|---|---|---|---|---|")
        for sys_name, res in report.items():
            sl = res.get(tn, {}).get("apc", {}).get("slices", {})
            def fmt(k):
                return f"{sl[k]['macro_f1']:.3f} (n={sl[k]['n']})" if k in sl else "-"
            L.append(
                f"| **{sys_name}** | {fmt('irony')} | {fmt('idiom')} | {fmt('gen_z')} | "
                f"{fmt('negation')} | {fmt('contrast')} | {fmt('implicit')} |"
            )
    L.append("\n**Data note:** metrics use separate public test sets; their annotation protocols differ. ")
    L.append("Implicit scores for baseline 3-way APC are N/A; it has no open-set aspect detection. ")
    return "\n".join(L)

def stage_export(a):
    from absa.infer import ABSAPipeline

    ate_src = C.MODELS / "ate_full" / "final"
    apc_src = C.MODELS / "apc_full" / "final"
    if not (ate_src.exists() and apc_src.exists()):
        print("[export] trained models not found", flush=True)
        return
    dst_ate = C.EXPORT / "ate_model"
    dst_apc = C.EXPORT / "apc_model"
    for s, d in ((ate_src, dst_ate), (apc_src, dst_apc)):
        if d.exists():
            shutil.rmtree(d)
        shutil.copytree(s, d)
    shutil.copy2(S / "implicit_ontology.json", dst_apc / "implicit_ontology.json")

    pipe = ABSAPipeline(str(dst_ate), str(dst_apc), device="cpu")
    lines = ["# Inference examples from the public held-out ABSA-TR test split (CPU)\n"]
    test_docs = read_jsonl(S / "absa_tr_test.jsonl")
    sample = random.Random(a.seed).sample(test_docs, min(6, len(test_docs)))
    for doc in sample:
        text = doc["text"]
        r = pipe.analyze(text, domain=doc.get("domain"))
        lines.append(f"**Source ID:** `{doc['id']}`  \n**Review:** `{text}`\n")
        for asp in r["aspects"]:
            tag = "IMPLICIT" if asp["implicit"] else f"SPAN[{asp['start']}:{asp['end']}]"
            lines.append(f"- **{asp['aspect']}** ({tag}) → **{asp['polarity'].upper()}** (conf={asp['confidence']:.2f})")
        lines.append("")
    demo_md = "\n".join(lines)
    (C.REPORTS / "demo.md").write_text(demo_md, encoding="utf-8")
    print(demo_md, flush=True)
    zp = shutil.make_archive(str(C.WORK / "turkish_absa_models"), "zip", C.EXPORT)
    print("[export] zip:", zp, flush=True)


## 3. Install dependencies and record the environment

The GPU is required for model training. The dependency list and an installed-package snapshot are saved with the output so the result can be reviewed later.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
import torch, transformers, huggingface_hub
print('torch:', torch.__version__, '| transformers:', transformers.__version__, '| hub:', huggingface_hub.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')
(WORK / 'reports').mkdir(parents=True, exist_ok=True)
with (WORK / 'reports' / 'environment.txt').open('w') as output:
    output.write(f'Python: {sys.version}\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}\n')
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=output, check=True)


In [ ]:
from run_public import stage_prepare, stage_clean, stage_eda, stage_train, stage_eval, stage_export
settings = SimpleNamespace(seed=SEED, quick=QUICK_MODE, no_clothing=False,
                           max_len=MAX_LEN, eda_tokenizer=True, epochs=EPOCHS,
                           bs=BATCH_SIZE, grad_accum=GRAD_ACCUM, lr=3.5e-5,
                           train_real_baseline=False, retrain=False,
                           skip_baseline=not RUN_BASELINE,
                           eval_implicit_limit=EVAL_IMPLICIT_LIMIT)
def run_stage(stage):
    print(f'===== {stage} =====', flush=True)
    globals()[f'stage_{stage}'](settings)


## 4. Download public data and record upstream revisions

The original split assignments are retained. The clothing dataset is used for APC only because it has no span annotations. Dataset revisions are recorded at run time. Internet access is needed for this step and the model downloads.


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repositories = [('dataset', 'ytu-ce-cosmos/absa-tr'),
                ('dataset', 'Multilingual-NLP/M-ABSA'),
                ('dataset', 'ebrukilic/clothing_products_ABSA'),
                ('model', 'ytu-ce-cosmos/modernbert-tr-base'),
                ('model', 'ytu-ce-cosmos/modernbert-tr-absa-extraction'),
                ('model', 'ytu-ce-cosmos/modernbert-tr-absa-polarity')]
revisions = {}
for kind, repo in repositories:
    info = api.dataset_info(repo) if kind == 'dataset' else api.model_info(repo)
    revisions[repo] = info.sha
(WORK / 'reports' / 'upstream_revisions.json').write_text(json.dumps(revisions, indent=2))
os.environ['ABSA_DATASET_REVISIONS'] = json.dumps({repo: revisions[repo] for kind, repo in repositories if kind == 'dataset'})
os.environ['ABSA_MODEL_REVISIONS'] = json.dumps({repo: revisions[repo] for kind, repo in repositories if kind == 'model'})
print(json.dumps(revisions, indent=2))
run_stage('prepare')
manifest = json.loads((WORK / 'data' / 'splits' / 'manifest.json').read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 5. Audit, conservative cleaning, and exploratory analysis

The audit checks span offsets, conflicting labels, and split overlap. Cleaning affects **training only**: it removes invalid spans, non-linguistic placeholders, normalized duplicates, normalized text collisions and conservative character five-gram near-duplicates against held-out data. The near-duplicate search is approximate. Documents with any unaligned explicit aspect are excluded as a whole, with counts recorded in the manifest. Regex diagnostics flag URLs, email, phone-like strings, markup and HTML entities, zero-width characters, repeated punctuation, elongated words, low letter ratios, and suspiciously long or short reviews. Flagged text stays in the data unless a separate, explicit rejection rule applies; changing review characters would break gold offsets.

EDA reports source/domain balance, polarity, aspect frequency, explicit versus implicit supervision, multi-aspect reviews, length quantiles, duplicate counts, ambiguous category polarity, and ModernBERT truncation risk. Inspect these reports before training.


In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'scripts' / 'audit_data.py'),
                '--work', str(WORK)], cwd=PROJECT, check=True)
audit = json.loads((WORK / 'reports' / 'data_audit.json').read_text())
print('Invalid gold spans:', len(audit['invalid_spans']))
print('Conflicting annotations flagged for review:', len(audit['conflicts']))
if audit['invalid_spans']:
    raise RuntimeError('Invalid gold offsets detected. Inspect data_audit.json before training.')
run_stage('clean')
run_stage('eda')
cleaning = json.loads((WORK / 'reports' / 'cleaning_report.json').read_text())
eda = json.loads((WORK / 'reports' / 'eda_report.json').read_text())
print('Cleaning counts:', cleaning['counts'])
print('Removal reasons:', cleaning['dropped_by_reason'])
print('Token profile:', eda['token_lengths_train_clean_plus_val'])
if eda['token_lengths_train_clean_plus_val']['status'] != 'ok':
    raise RuntimeError('Exact tokenizer EDA failed; inspect eda_report.json.')
if any(row['normalized_text_collisions'] for row in eda['normalized_cross_split_overlap']
       if row['train'] == 'train_clean'):
    raise RuntimeError('Cleaned training data still overlaps a held-out split.')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

reports = WORK / 'reports'
summary = pd.read_csv(reports / 'eda_summary.csv')
sources = pd.read_csv(reports / 'eda_source.csv')
polarity = pd.read_csv(reports / 'eda_polarity.csv')
domains = pd.read_csv(reports / 'eda_domain.csv')
noise = pd.read_csv(reports / 'eda_noise.csv')
display(summary)
display(sources.pivot_table(index='split', columns='source', values='documents', fill_value=0))
display(domains[domains['split'] == 'train_clean'].sort_values('documents', ascending=False).head(20))
display(noise[noise['split'] == 'train_public'].sort_values('documents', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sources[sources['split'] == 'train_clean'].set_index('source')['documents'].sort_values().plot.barh(ax=axes[0], title='Clean training documents by source')
polarity[polarity['split'] == 'train_clean'].set_index('polarity')['aspects'].plot.bar(ax=axes[1], title='Clean training aspect polarity')
for ax in axes: ax.set_xlabel('Count')
fig.tight_layout()
fig.savefig(reports / 'eda_overview.png', dpi=160, bbox_inches='tight')
plt.show()


## 6. Train ATE and APC

ATE learns BIO tags from span-supervised documents. APC learns sentiment for a `(review, aspect)` pair with a distinct `not_mentioned` class; this prevents treating an absent aspect as neutral. Ambiguous same-target polarity views are excluded from APC training. The candidate ontology is learned from cleaned training data by domain. Long reviews use overlapping token windows. APC checkpoint selection uses validation macro F1 on gold pairs under the same target policy as final testing; heuristic absent-aspect negatives remain in validation but are reported separately. Models are trained sequentially on one GPU or in parallel when two GPUs are available. Test splits are reserved for final evaluation.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before training.')
run_stage('train')
for task in ('ate', 'apc'):
    path = WORK / 'models' / f'{task}_full' / 'train_summary.json'
    print(task.upper(), json.loads(path.read_text())['val'])


## 7. Validation error analysis

Use validation errors to diagnose target normalization, negation, and domain mismatch. High-confidence errors are candidates for manual review. This cell does not modify labels or tune on the test sets.


In [ ]:
from absa.data import read_jsonl
from absa.infer import APCPredictor
from absa import config as C
from run_public import _gold_apc_pairs
from sklearn.metrics import confusion_matrix

validation = read_jsonl(WORK / 'data' / 'splits' / 'val.jsonl')
texts, targets, gold_ids, metadata = _gold_apc_pairs(validation)
apc = APCPredictor(str(WORK / 'models' / 'apc_full' / 'final'))
pred_ids, confidence, _ = apc.classify_pairs(texts, targets)
gold_names = [C.APC_I2L[i] for i in gold_ids]
pred_names = [apc.label_names[i] for i in pred_ids]
labels = ['negative', 'neutral', 'positive', 'not_mentioned']
display(pd.DataFrame(confusion_matrix(gold_names, pred_names, labels=labels),
                     index=[f'gold:{x}' for x in labels],
                     columns=[f'pred:{x}' for x in labels]))
errors = pd.DataFrame([{'source_id': m['id'], 'text': t, 'target': a,
                        'gold': g, 'prediction': p, 'confidence': round(c, 3)}
                       for t, a, g, p, c, m in zip(texts, targets, gold_names, pred_names, confidence, metadata)
                       if g != p], columns=['source_id', 'text', 'target', 'gold', 'prediction', 'confidence'])
errors = errors.sort_values('confidence', ascending=False)
errors.to_csv(reports / 'apc_validation_errors.csv', index=False)
display(errors.head(15))
del apc


## 8. Final held-out evaluation

Report ATE exact and overlap F1, APC macro F1 on gold aspects, and end-to-end explicit tuple F1 separately. The implicit score uses a uniform document sample and a training-derived domain ontology, so include sample size, positive prevalence, and gold ontology coverage. The clothing test has no ATE/E2E score. Do not combine scores from annotation protocols into one unexplained number.


In [ ]:
run_stage('eval')
results = json.loads((reports / 'results.json').read_text())
from IPython.display import Markdown
display(Markdown((reports / 'results.md').read_text()))
rows = []
for system, tests in results.items():
    for test_name, metrics in tests.items():
        rows.append({'system': system, 'test': test_name,
                     'ATE_exact_F1': metrics['ate']['exact']['f1'] if metrics['ate'] else None,
                     'APC_gold_macro_F1': metrics['apc']['macro_f1'],
                     'E2E_overlap_F1': metrics['e2e']['overlap_f1'] if metrics['e2e'] else None,
                     'implicit_subset_F1': metrics['implicit']['f1'] if metrics.get('implicit') else None})
display(pd.DataFrame(rows).round(3))


## 9. Export models and a reviewable report bundle

The demo selects examples from a public held-out split at a fixed seed; it is an illustration, not an additional benchmark. The report bundle contains environment details, revision identifiers, data quality reports, EDA tables, validation errors, training summaries, and final evaluation.


In [ ]:
run_stage('export')
display(Markdown((reports / 'demo.md').read_text()))
from zipfile import ZipFile, ZIP_DEFLATED
bundle = WORK / 'turkish_absa_report_bundle.zip'
include = ['data/splits/manifest.json', 'reports/environment.txt',
           'reports/upstream_revisions.json', 'reports/data_audit.json',
           'reports/review_queue.jsonl',
           'reports/cleaning_report.json', 'reports/eda_report.json',
           'reports/eda_overview.png', 'reports/apc_validation_errors.csv',
           'reports/results.json', 'reports/results.md', 'reports/demo.md',
           'models/ate_full/train_summary.json', 'models/apc_full/train_summary.json']
include += [str(p.relative_to(WORK)) for p in reports.glob('eda_*.csv')]
with ZipFile(bundle, 'w', ZIP_DEFLATED) as archive:
    for rel in include:
        path = WORK / rel
        if path.exists(): archive.write(path, rel)
print('Model archive:', WORK / 'turkish_absa_models.zip')
print('Report bundle:', bundle)


## Reporting guidance

State the exact data counts, alignment and cleaning exclusions, held-out scores per dataset, GPU and package versions, seed, and known annotation limitations. The ATE threshold remains fixed and uncalibrated. Improvement claims require repeated seeds, paired uncertainty estimates, a controlled ablation, and independent review of difficult Turkish expressions. Do not present a quick run or validation-selected score as a final test result.
